# 04 - Customer Feature Engineering

## Objective

This notebook transforms the cleaned transaction-level retail dataset into a
customer-level feature dataset for clustering.

The aim is to create one record per customer describing meaningful purchasing
behaviour.

Initial feature areas will include:

- Recency - how recently the customer purchased;
- Frequency - how often the customer purchased;
- Monetary Value - how much the customer spent;
- Average Order Value;
- Product Diversity;
- Customer Tenure;
- Postage-related behaviour.

Feature distributions, skewness, outliers and scaling will be assessed in the
next preprocessing stage before clustering.

In [2]:
# necessary libraries
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import Data
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import AccountKeyConfiguration
import mltable
from mltable import MLTableHeaders, MLTableFileEncoding
import pandas as pd
import numpy as np

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

# Load the registered Azure ML Data Asset so that analysis is based on the
# governed MLTable rather than a local file path.
paid_purchase_asset = ml_client.data.get(
    name="online-retail-paid-purchases",
    version="1"
)

# Load the MLTable definition and materialise the dataset as a Pandas DataFrame
# for interactive profiling and exploratory analysis.
retail_table = mltable.load(paid_purchase_asset.path)
df_paid = retail_table.to_pandas_dataframe()

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [3]:
# Validate the size of the cleaned dataset before feature engineering begins.

print("Rows:", f"{len(df_paid):,}")
print("Columns:", df_paid.shape[1])

df_paid.head()

Rows: 392,692
Columns: 10


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionCategory,LineValue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,Merchandise,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Merchandise,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,Merchandise,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Merchandise,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Merchandise,20.34


## 1. Core RFM Features

Recency, Frequency and Monetary Value provide a baseline representation of
customer purchasing behaviour.

For the core RFM calculation, only transactions classified as `Merchandise`
are used.

Non-merchandise activity such as postage, bank charges and manual transactions
has been retained in the dataset and will be engineered separately as
additional behavioural features.

The RFM features are defined as:

- **Recency:** Number of days since the customer's most recent purchase.
- **Frequency:** Number of unique purchase invoices placed by the customer.
- **Monetary Value:** Total merchandise value purchased by the customer.

In [4]:
# Restrict the core RFM calculation to standard merchandise purchases.
# Non-merchandise transactions remain available in df_paid for separate
# behavioural feature engineering later.

df_merchandise = df_paid.loc[
    df_paid["TransactionCategory"] == "Merchandise"
].copy()

print(f"Merchandise transaction rows: {len(df_merchandise):,}")
print(f"Unique merchandise customers: {df_merchandise['CustomerID'].nunique():,}")

Merchandise transaction rows: 391,283
Unique merchandise customers: 4,334


In [5]:
# Define a consistent reference date for calculating customer recency.
# One day is added so that a customer purchasing on the final dataset date
# has a Recency value of 1 rather than 0.

reference_date = (
    df_merchandise["InvoiceDate"].max()
    + pd.Timedelta(days=1)
)

print("Latest transaction date:", df_merchandise["InvoiceDate"].max())
print("Recency reference date:", reference_date)

Latest transaction date: 2011-12-09 12:50:00
Recency reference date: 2011-12-10 12:50:00


In [6]:
# Find each customer's most recent merchandise purchase date.

customer_recency = (
    df_merchandise
    .groupby("CustomerID")["InvoiceDate"]
    .max()
    .reset_index(name="LastPurchaseDate")
)

# Calculate the number of days between the reference date and each
# customer's most recent purchase.

customer_recency["Recency"] = (
    reference_date - customer_recency["LastPurchaseDate"]
).dt.days

customer_recency.head()

,CustomerID,LastPurchaseDate,Recency
0,12346.0,2011-01-18 10:01:00,326
1,12347.0,2011-12-07 15:52:00,2
2,12348.0,2011-09-25 13:13:00,75
3,12349.0,2011-11-21 09:51:00,19
4,12350.0,2011-02-02 16:01:00,310


In [7]:
# Calculate purchase frequency as the number of unique merchandise invoices
# associated with each customer.

customer_frequency = (
    df_merchandise
    .groupby("CustomerID")["InvoiceNo"]
    .nunique()
    .reset_index(name="Frequency")
)

customer_frequency.head()

,CustomerID,Frequency
0,12346.0,1
1,12347.0,7
2,12348.0,4
3,12349.0,1
4,12350.0,1


In [9]:
# Calculate the total value of merchandise purchased by each customer.

customer_monetary = (
    df_merchandise
    .groupby("CustomerID")["LineValue"]
    .sum()
    .reset_index(name="MonetaryValue")
)

customer_monetary.head()

,CustomerID,MonetaryValue
0,12346.0,77183.60
1,12347.0,4310.00
2,12348.0,1437.24
3,12349.0,1457.55
4,12350.0,294.40


In [10]:
# Combine Recency, Frequency and Monetary Value into one customer-level
# feature table.

customer_rfm = (
    customer_recency
    .merge(
        customer_frequency,
        on="CustomerID",
        how="inner"
    )
    .merge(
        customer_monetary,
        on="CustomerID",
        how="inner"
    )
)

customer_rfm.head()

,CustomerID,LastPurchaseDate,Recency,Frequency,MonetaryValue
0,12346.0,2011-01-18 10:01:00,326,1,77183.60
1,12347.0,2011-12-07 15:52:00,2,7,4310.00
2,12348.0,2011-09-25 13:13:00,75,4,1437.24
3,12349.0,2011-11-21 09:51:00,19,1,1457.55
4,12350.0,2011-02-02 16:01:00,310,1,294.40


In [12]:
customer_rfm.head(30)

,CustomerID,LastPurchaseDate,Recency,Frequency,MonetaryValue
0,12346.0,2011-01-18 10:01:00,326,1,77183.60
1,12347.0,2011-12-07 15:52:00,2,7,4310.00
2,12348.0,2011-09-25 13:13:00,75,4,1437.24
3,12349.0,2011-11-21 09:51:00,19,1,1457.55
4,12350.0,2011-02-02 16:01:00,310,1,294.40
5,12352.0,2011-11-03 14:37:00,36,7,1385.74
6,12353.0,2011-05-19 17:47:00,204,1,89.00
7,12354.0,2011-04-21 13:11:00,232,1,1079.40
8,12355.0,2011-05-09 13:49:00,214,1,459.40
9,12356.0,2011-11-17 08:40:00,23,3,2487.43


In [13]:
# Confirm that the RFM table contains one row per customer.

print("RFM shape:", customer_rfm.shape)
print("Unique customers:", customer_rfm["CustomerID"].nunique())

assert len(customer_rfm) == customer_rfm["CustomerID"].nunique()

print("Customer-level uniqueness check passed.")

RFM shape: (4334, 5)
Unique customers: 4334
Customer-level uniqueness check passed.


In [15]:
# Review the initial distributions of the core RFM features.
# Transformations and skewness treatment will be handled later during
# feature preprocessing.

customer_rfm[
    ["Recency", "Frequency", "MonetaryValue"]
].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Recency,4334.0,92.70,100.18,1.00,18.0,51.00,143.00,374.00
Frequency,4334.0,4.25,7.64,1.00,1.0,2.00,5.00,206.00
MonetaryValue,4334.0,2017.52,8920.36,3.75,304.3,663.71,1631.62,279138.02


### Initial RFM observations

The RFM summary indicates substantial variation in customer purchasing behaviour.

- The median customer made their most recent purchase 51 days before the
  reference date, compared with a mean Recency of approximately 93 days.
  This suggests a long tail of less recently active customers.

- The median customer placed 2 unique orders, while the maximum Frequency is
  206 orders, indicating a strongly right-skewed distribution.

- Median merchandise spend is approximately £664, compared with a mean of
  approximately £2,018 and a maximum exceeding £279,000. Monetary Value is
  therefore also likely to be strongly right-skewed and influenced by a
  relatively small number of high-value customers.

No transformation or outlier treatment is applied at this stage. Feature
distribution, skewness and extreme values will be assessed during the feature
preprocessing stage after the complete customer feature set has been created.

In [16]:
# Calculate Average Order Value as total merchandise spend divided by
# the number of unique merchandise purchase invoices.

customer_rfm["AverageOrderValue"] = (
    customer_rfm["MonetaryValue"]
    / customer_rfm["Frequency"]
)

In [18]:
# Review the distribution of customer Average Order Value.

customer_rfm["AverageOrderValue"].describe().T.round(2)

count     4334.00
mean       415.67
std       1800.86
min          3.75
25%        177.20
50%        289.77
75%        423.77
max      84236.25
Name: AverageOrderValue, dtype: float64

## 2. Product Diversity

Product Diversity measures the breadth of a customer's purchasing behaviour.

It is calculated as the number of unique merchandise `StockCode` values
purchased by each customer.

Only standard merchandise transactions are used so that postage, bank charges,
manual transactions and other non-merchandise activity do not artificially
increase the number of products purchased.

In [19]:
# Calculate the number of unique merchandise products purchased by each customer.

customer_product_diversity = (
    df_merchandise
    .groupby("CustomerID")["StockCode"]
    .nunique()
    .reset_index(name="UniqueProducts")
)

customer_product_diversity.head()

,CustomerID,UniqueProducts
0,12346.0,1
1,12347.0,103
2,12348.0,21
3,12349.0,72
4,12350.0,16


In [20]:
# Add Product Diversity to the existing customer-level feature dataset.

customer_features = customer_rfm.merge(
    customer_product_diversity,
    on="CustomerID",
    how="left"
)

In [21]:
# Confirm that the feature table still contains one row per customer.

print("Customer feature shape:", customer_features.shape)
print("Unique customers:", customer_features["CustomerID"].nunique())

customer_features.head()

Customer feature shape: (4334, 7)
Unique customers: 4334


,CustomerID,LastPurchaseDate,Recency,Frequency,MonetaryValue,AverageOrderValue,UniqueProducts
0,12346.0,2011-01-18 10:01:00,326,1,77183.60,77183.600000,1
1,12347.0,2011-12-07 15:52:00,2,7,4310.00,615.714286,103
2,12348.0,2011-09-25 13:13:00,75,4,1437.24,359.310000,21
3,12349.0,2011-11-21 09:51:00,19,1,1457.55,1457.550000,72
4,12350.0,2011-02-02 16:01:00,310,1,294.40,294.400000,16


In [22]:
# Review the distribution of Product Diversity.

customer_features["UniqueProducts"].describe().round(2)

count    4334.00
mean       61.43
std        85.31
min         1.00
25%        16.00
50%        35.00
75%        77.00
max      1786.00
Name: UniqueProducts, dtype: float64

## 3. Customer Tenure

Customer Tenure measures how long each customer has been active within the
available transaction history.

It is calculated as the number of days between the customer's first and most
recent merchandise purchase.

This helps distinguish long-standing customers from newer customers whose
purchase frequency may be lower simply because they have had less time to
transact.

In [23]:
# Identify each customer's first recorded merchandise purchase date.

customer_first_purchase = (
    df_merchandise
    .groupby("CustomerID")["InvoiceDate"]
    .min()
    .reset_index(name="FirstPurchaseDate")
)

customer_first_purchase.head()

,CustomerID,FirstPurchaseDate
0,12346.0,2011-01-18 10:01:00
1,12347.0,2010-12-07 14:57:00
2,12348.0,2010-12-16 19:09:00
3,12349.0,2011-11-21 09:51:00
4,12350.0,2011-02-02 16:01:00


In [24]:
# Add the first purchase date to the customer-level feature dataset.

customer_features = customer_features.merge(
    customer_first_purchase,
    on="CustomerID",
    how="left"
)

In [25]:
# Calculate customer tenure as the number of days between the first and
# most recent merchandise purchase.

customer_features["CustomerTenureDays"] = (
    customer_features["LastPurchaseDate"]
    - customer_features["FirstPurchaseDate"]
).dt.days

In [26]:
# Review the distribution of customer tenure.

customer_features["CustomerTenureDays"].describe().round(2)

count    4334.00
mean      130.29
std       132.09
min         0.00
25%         0.00
50%        92.00
75%       252.00
max       373.00
Name: CustomerTenureDays, dtype: float64

In [27]:
# Count customers whose observed purchasing history spans only one day.

single_day_customers = (
    customer_features["CustomerTenureDays"] == 0
).sum()

print(
    f"Customers with zero-day observed tenure: "
    f"{single_day_customers:,}"
)

Customers with zero-day observed tenure: 1,564


In [28]:
customer_features.head()

,CustomerID,LastPurchaseDate,Recency,Frequency,MonetaryValue,AverageOrderValue,UniqueProducts,FirstPurchaseDate,CustomerTenureDays
0,12346.0,2011-01-18 10:01:00,326,1,77183.60,77183.600000,1,2011-01-18 10:01:00,0
1,12347.0,2011-12-07 15:52:00,2,7,4310.00,615.714286,103,2010-12-07 14:57:00,365
2,12348.0,2011-09-25 13:13:00,75,4,1437.24,359.310000,21,2010-12-16 19:09:00,282
3,12349.0,2011-11-21 09:51:00,19,1,1457.55,1457.550000,72,2011-11-21 09:51:00,0
4,12350.0,2011-02-02 16:01:00,310,1,294.40,294.400000,16,2011-02-02 16:01:00,0


## 4. Total Merchandise Quantity

Total Merchandise Quantity measures the total number of merchandise units
purchased by each customer.

This feature complements Monetary Value by distinguishing customers who purchase
large quantities of lower-value products from customers who purchase smaller
quantities of higher-value products.

In [29]:
# Calculate the total quantity of merchandise purchased by each customer.

customer_quantity = (
    df_merchandise
    .groupby("CustomerID")["Quantity"]
    .sum()
    .reset_index(name="TotalQuantity")
)

customer_quantity.head()

,CustomerID,TotalQuantity
0,12346.0,74215
1,12347.0,2458
2,12348.0,2332
3,12349.0,630
4,12350.0,196


In [30]:
# Add Total Merchandise Quantity to the customer-level feature dataset.

customer_features = customer_features.merge(
    customer_quantity,
    on="CustomerID",
    how="left"
)

In [31]:
# Review the distribution of total merchandise quantity purchased.

customer_features["TotalQuantity"].describe().round(2)

count      4334.00
mean       1186.41
std        5040.96
min           1.00
25%         159.25
50%         377.50
75%         989.75
max      196844.00
Name: TotalQuantity, dtype: float64

## 5. Average Quantity per Order

Average Quantity per Order measures the typical number of merchandise units
purchased by a customer per order.

It is calculated by dividing the customer's total merchandise quantity by the
number of unique merchandise invoices.

This helps distinguish customers who place small orders from customers who
typically purchase larger volumes per transaction.

In [32]:
# Calculate the average number of merchandise units purchased per order.

customer_features["AverageQuantityPerOrder"] = (
    customer_features["TotalQuantity"]
    / customer_features["Frequency"]
)

In [33]:
# Review the distribution of average merchandise quantity per order.

customer_features["AverageQuantityPerOrder"].describe().round(2)

count     4334.00
mean       255.25
std       1316.36
min          1.00
25%         92.52
50%        161.00
75%        271.98
max      74215.00
Name: AverageQuantityPerOrder, dtype: float64

## 6. Average Days Between Purchases

Average Days Between Purchases measures the typical time interval between
successive merchandise orders for repeat customers.

The calculation is performed at invoice level so that multiple product lines
within the same order are not treated as separate purchases.

For customers with only one observed purchase, the feature is undefined and
will remain missing at this stage. Missing-value treatment will be considered
during feature preprocessing.

In [34]:
# Create an invoice-level dataset so that each customer order is represented
# once, regardless of how many merchandise lines the invoice contains.

customer_invoices = (
    df_merchandise
    .groupby(["CustomerID", "InvoiceNo"])["InvoiceDate"]
    .min()
    .reset_index()
    .sort_values(["CustomerID", "InvoiceDate"])
)

customer_invoices.head()

,CustomerID,InvoiceNo,InvoiceDate
0,12346.0,541431,2011-01-18 10:01:00
1,12347.0,537626,2010-12-07 14:57:00
2,12347.0,542237,2011-01-26 14:30:00
3,12347.0,549222,2011-04-07 10:43:00
4,12347.0,556201,2011-06-09 13:01:00


In [36]:
# Calculate the elapsed time between consecutive merchandise orders
# for each customer.

customer_invoices["DaysSincePreviousPurchase"] = (
    customer_invoices
    .groupby("CustomerID")["InvoiceDate"]
    .diff()
    .dt.total_seconds()
    / 86400 # (number of seconds in a day)
)

customer_invoices.head()

,CustomerID,InvoiceNo,InvoiceDate,DaysSincePreviousPurchase
0,12346.0,541431,2011-01-18 10:01:00,NaN
1,12347.0,537626,2010-12-07 14:57:00,NaN
2,12347.0,542237,2011-01-26 14:30:00,49.981250
3,12347.0,549222,2011-04-07 10:43:00,70.842361
4,12347.0,556201,2011-06-09 13:01:00,63.095833


In [38]:
# Calculate the mean number of days between successive orders
# for each repeat customer.

customer_purchase_interval = (
    customer_invoices
    .groupby("CustomerID")["DaysSincePreviousPurchase"]
    .mean()
    .reset_index(name="AverageDaysBetweenPurchases")
)

customer_purchase_interval.head(10)

,CustomerID,AverageDaysBetweenPurchases
0,12346.0,NaN
1,12347.0,60.839699
2,12348.0,94.250926
3,12349.0,NaN
4,12350.0,NaN
5,12352.0,43.347685
6,12353.0,NaN
7,12354.0,NaN
8,12355.0,NaN
9,12356.0,151.475694


In [39]:
# Add the average purchase interval to the customer-level feature dataset.

customer_features = customer_features.merge(
    customer_purchase_interval,
    on="CustomerID",
    how="left"
)

In [40]:
customer_features["AverageDaysBetweenPurchases"].describe().round(2)

count    2829.00
mean       73.10
std        65.54
min         0.00
25%        30.08
50%        54.05
75%        92.89
max       365.98
Name: AverageDaysBetweenPurchases, dtype: float64

In [41]:
# Customers with only one purchase do not have an observable interval
# between purchases.

missing_purchase_interval = (
    customer_features["AverageDaysBetweenPurchases"]
    .isna()
    .sum()
)

print(
    f"Customers without a purchase interval: "
    f"{missing_purchase_interval:,}"
)

Customers without a purchase interval: 1,505


In [42]:
customer_features.head()

,CustomerID,LastPurchaseDate,Recency,Frequency,MonetaryValue,AverageOrderValue,UniqueProducts,FirstPurchaseDate,CustomerTenureDays,TotalQuantity,AverageQuantityPerOrder,AverageDaysBetweenPurchases
0,12346.0,2011-01-18 10:01:00,326,1,77183.60,77183.600000,1,2011-01-18 10:01:00,0,74215,74215.000000,NaN
1,12347.0,2011-12-07 15:52:00,2,7,4310.00,615.714286,103,2010-12-07 14:57:00,365,2458,351.142857,60.839699
2,12348.0,2011-09-25 13:13:00,75,4,1437.24,359.310000,21,2010-12-16 19:09:00,282,2332,583.000000,94.250926
3,12349.0,2011-11-21 09:51:00,19,1,1457.55,1457.550000,72,2011-11-21 09:51:00,0,630,630.000000,NaN
4,12350.0,2011-02-02 16:01:00,310,1,294.40,294.400000,16,2011-02-02 16:01:00,0,196,196.000000,NaN


### Purchase interval observation

A total of 1,505 customers do not have an observable purchase interval because
they made only one merchandise purchase during the available transaction period.

These missing values are therefore structurally meaningful rather than data
quality issues. They will remain missing at this stage and will be handled
during feature preprocessing.

The result also indicates that approximately 35% of customers are
single-purchase customers, which may represent an important behavioural group
for segmentation.

In [43]:
# Identify whether each customer has made more than one merchandise purchase.

customer_features["RepeatCustomer"] = (
    customer_features["Frequency"] > 1
).astype(int)

In [44]:
customer_features["RepeatCustomer"].value_counts()

RepeatCustomer
1    2829
0    1505
Name: count, dtype: int64

In [45]:
customer_features.head()

,CustomerID,LastPurchaseDate,Recency,Frequency,MonetaryValue,AverageOrderValue,UniqueProducts,FirstPurchaseDate,CustomerTenureDays,TotalQuantity,AverageQuantityPerOrder,AverageDaysBetweenPurchases,RepeatCustomer
0,12346.0,2011-01-18 10:01:00,326,1,77183.60,77183.600000,1,2011-01-18 10:01:00,0,74215,74215.000000,NaN,0
1,12347.0,2011-12-07 15:52:00,2,7,4310.00,615.714286,103,2010-12-07 14:57:00,365,2458,351.142857,60.839699,1
2,12348.0,2011-09-25 13:13:00,75,4,1437.24,359.310000,21,2010-12-16 19:09:00,282,2332,583.000000,94.250926,1
3,12349.0,2011-11-21 09:51:00,19,1,1457.55,1457.550000,72,2011-11-21 09:51:00,0,630,630.000000,NaN,0
4,12350.0,2011-02-02 16:01:00,310,1,294.40,294.400000,16,2011-02-02 16:01:00,0,196,196.000000,NaN,0


## 7. Postage Behaviour

Postage transactions were retained during cleaning because they may provide
useful information about customer purchasing and fulfilment behaviour.

Postage is excluded from core merchandise RFM measures but is used to derive
separate customer-level features:

- **PostageSpend:** Total amount spent on postage.
- **PostageInvoices:** Number of invoices containing a postage charge.
- **PostageInvoiceShare:** Proportion of the customer's paid invoices that
  contain a postage charge.
- **HasPostage:** Indicates whether the customer has incurred any postage charge.

In [46]:
# Select paid transaction lines classified as postage.

df_postage = df_paid.loc[
    df_paid["TransactionCategory"] == "Postage"
].copy()

print(f"Postage rows: {len(df_postage):,}")
print(f"Customers with postage: {df_postage['CustomerID'].nunique():,}")

Postage rows: 1,115
Customers with postage: 332


In [47]:
# Calculate the total monetary value of postage charged to each customer.

customer_postage_spend = (
    df_postage
    .groupby("CustomerID")["LineValue"]
    .sum()
    .reset_index(name="PostageSpend")
)

In [48]:
# Count the number of unique invoices containing a postage charge
# for each customer.

customer_postage_invoices = (
    df_postage
    .groupby("CustomerID")["InvoiceNo"]
    .nunique()
    .reset_index(name="PostageInvoices")
)

In [49]:
# Combine the postage measures into one customer-level table.

customer_postage = customer_postage_spend.merge(
    customer_postage_invoices,
    on="CustomerID",
    how="outer"
)

customer_postage.head()

,CustomerID,PostageSpend,PostageInvoices
0,12348.0,360.0,4
1,12349.0,300.0,1
2,12350.0,40.0,1
3,12352.0,280.0,5
4,12356.0,324.0,1


In [50]:
# Add postage behaviour to the main customer feature dataset.

customer_features = customer_features.merge(
    customer_postage,
    on="CustomerID",
    how="left"
)

In [51]:
# Customers without postage transactions have genuine zero postage activity.

customer_features[
    ["PostageSpend", "PostageInvoices"]
] = customer_features[
    ["PostageSpend", "PostageInvoices"]
].fillna(0)

In [52]:
customer_features[
    ["PostageSpend", "PostageInvoices"]
].describe().round(2)

,PostageSpend,PostageInvoices
count,4334.00,4334.00
mean,20.70,0.26
std,238.34,1.42
min,0.00,0.00
25%,0.00,0.00
50%,0.00,0.00
75%,0.00,0.00
max,11906.36,31.00


### Postage Invoice Share

Postage Invoice Share measures the proportion of a customer's paid invoices
that contain a postage charge.

The denominator is calculated from all paid invoices rather than merchandise
Frequency, because postage and other paid non-merchandise transactions were
retained separately from the core RFM calculation.

In [53]:
# Count the total number of unique paid invoices for each customer.

customer_paid_invoices = (
    df_paid
    .groupby("CustomerID")["InvoiceNo"]
    .nunique()
    .reset_index(name="PaidInvoices")
)

customer_paid_invoices.head()

,CustomerID,PaidInvoices
0,12346.0,1
1,12347.0,7
2,12348.0,4
3,12349.0,1
4,12350.0,1


In [54]:
# Add the total number of paid invoices to the customer-level feature table.

customer_features = customer_features.merge(
    customer_paid_invoices,
    on="CustomerID",
    how="left"
)

In [55]:
# Calculate the proportion of paid invoices that contained postage.

customer_features["PostageInvoiceShare"] = (
    customer_features["PostageInvoices"]
    / customer_features["PaidInvoices"]
)

In [56]:
# Flag whether the customer has ever incurred a postage charge.

customer_features["HasPostage"] = (
    customer_features["PostageInvoices"] > 0
).astype(int)

In [58]:
customer_features[
    [
        "PaidInvoices",
        "PostageInvoices",
        "PostageInvoiceShare",
        "HasPostage"
    ]
].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
PaidInvoices,4334.0,4.28,7.70,1.0,1.0,2.0,5.0,209.0
PostageInvoices,4334.0,0.26,1.42,0.0,0.0,0.0,0.0,31.0
PostageInvoiceShare,4334.0,0.06,0.23,0.0,0.0,0.0,0.0,1.0
HasPostage,4334.0,0.08,0.27,0.0,0.0,0.0,0.0,1.0


In [59]:
# PostageInvoiceShare must always fall between 0 and 1.

assert customer_features["PostageInvoiceShare"].between(0, 1).all()

print("Postage invoice share validation passed.")

Postage invoice share validation passed.


In [60]:
# Retrieve the registered returns dataset from Azure ML.

returns_asset = ml_client.data.get(
    name="online-retail-returns",
    version="1"
)

print(returns_asset.name)
print(returns_asset.version)

online-retail-returns
1


In [61]:
# Load the returns MLTable and materialise it as a Pandas DataFrame.

returns_table = mltable.load(
    returns_asset.path
)

df_returns = returns_table.to_pandas_dataframe()

In [62]:
print("Returns rows:", f"{len(df_returns):,}")
print("Returns columns:", df_returns.shape[1])

df_returns.head()

Returns rows: 8,872
Returns columns: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
1,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
2,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
3,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
4,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


## 8. Return and Cancellation Behaviour

Return behaviour is derived from the cancelled transaction dataset retained
during the cleaning stage.

Because a single cancelled invoice may contain multiple product lines, return
frequency is measured using the number of unique cancelled invoices rather
than the number of transaction rows.

In [63]:
# Count the number of unique cancelled invoices associated with each customer.

customer_returns = (
    df_returns
    .groupby("CustomerID")["InvoiceNo"]
    .nunique()
    .reset_index(name="ReturnInvoices")
)

customer_returns.head()

,CustomerID,ReturnInvoices
0,12346.0,1
1,12352.0,3
2,12359.0,2
3,12362.0,3
4,12365.0,1


In [64]:
# Add return frequency to the customer-level feature dataset.

customer_features = customer_features.merge(
    customer_returns,
    on="CustomerID",
    how="left"
)

In [65]:
# Customers without a matching cancellation record have zero observed returns.

customer_features["ReturnInvoices"] = (
    customer_features["ReturnInvoices"]
    .fillna(0)
    .astype(int)
)

In [66]:
customer_features["ReturnInvoices"].describe().round(2)

count    4334.00
mean        0.83
std         2.15
min         0.00
25%         0.00
50%         0.00
75%         1.00
max        47.00
Name: ReturnInvoices, dtype: float64

In [67]:
print(
    "Customers with at least one return:",
    (customer_features["ReturnInvoices"] > 0).sum()
)

Customers with at least one return: 1553


### Return Value

Return Value measures the total monetary value associated with a customer's
cancelled or returned transaction lines.

Cancellation quantities are recorded as negative values in the source data.
The absolute transaction value is therefore used so that Return Value represents
the magnitude of returned or cancelled activity rather than a negative monetary
amount.

This feature is kept separate from merchandise Monetary Value and will later
support measures such as customer return rate.

In [68]:
# Calculate the positive monetary magnitude of each cancelled transaction line.

df_returns["ReturnLineValue"] = (
    df_returns["Quantity"] * df_returns["UnitPrice"]
).abs()

In [69]:
df_returns[
    ["InvoiceNo", "Quantity", "UnitPrice", "ReturnLineValue"]
].head()

,InvoiceNo,Quantity,UnitPrice,ReturnLineValue
0,C536379,-1,27.50,27.50
1,C536383,-1,4.65,4.65
2,C536391,-12,1.65,19.80
3,C536391,-24,0.29,6.96
4,C536391,-24,0.29,6.96


In [70]:
# Calculate the total monetary value of cancelled or returned activity
# associated with each customer.

customer_return_value = (
    df_returns
    .groupby("CustomerID")["ReturnLineValue"]
    .sum()
    .reset_index(name="ReturnValue")
)

In [71]:
# Add return monetary value to the customer-level feature dataset.

customer_features = customer_features.merge(
    customer_return_value,
    on="CustomerID",
    how="left"
)

In [72]:
customer_features["ReturnValue"] = (
    customer_features["ReturnValue"]
    .fillna(0)
)

In [73]:
customer_features["ReturnValue"].describe().round(2)

count      4334.00
mean        137.53
std        2959.24
min           0.00
25%           0.00
50%           0.00
75%          15.90
max      168469.60
Name: ReturnValue, dtype: float64

### Review of Return Transaction Types

Before calculating a merchandise return rate, cancelled transactions are
reviewed to distinguish merchandise returns from non-merchandise adjustments
such as discounts or administrative transactions.

This ensures that the return-value numerator is comparable with the merchandise
Monetary Value used in the denominator.

In [74]:
# Identify return StockCodes that do not contain a numeric character.
# These are reviewed separately because they may represent discounts,
# postage or other non-merchandise transaction types.

non_standard_return_mask = (
    ~df_returns["StockCode"]
    .astype("string")
    .str.contains(r"\d", na=False)
)

non_standard_returns = (
    df_returns.loc[non_standard_return_mask]
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .agg(
        row_count=("InvoiceNo", "size"),
        unique_invoices=("InvoiceNo", "nunique"),
        unique_customers=("CustomerID", "nunique"),
        total_return_value=("ReturnLineValue", "sum")
    )
    .reset_index()
    .sort_values("total_return_value", ascending=False)
)

non_standard_returns

,StockCode,Description,row_count,unique_invoices,unique_customers,total_return_value
2,M,Manual,175,154,112,112165.39
3,POST,POSTAGE,97,95,85,11093.72
0,CRUK,CRUK Commission,16,16,1,7933.43
1,D,Discount,77,65,24,5696.22


### Merchandise Return Classification

The review identified four non-merchandise cancellation types: manual
adjustments, postage, commission and discounts.

These transactions are retained for traceability but excluded from the
merchandise return-value calculation because they do not represent returned
products.

A separate merchandise return measure is therefore created for comparison with
customer merchandise spend.

In [75]:
# Classify known non-merchandise cancellation codes while treating
# remaining product codes as merchandise returns.

non_merchandise_return_categories = {
    "M": "Manual transaction",
    "POST": "Postage",
    "CRUK": "Commission",
    "D": "Discount"
}

df_returns["ReturnCategory"] = (
    df_returns["StockCode"]
    .map(non_merchandise_return_categories)
    .fillna("Merchandise")
)

In [76]:
df_returns["ReturnCategory"].value_counts()

ReturnCategory
Merchandise           8507
Manual transaction     175
Postage                 97
Discount                77
Commission              16
Name: count, dtype: int64

In [77]:
# Retain only product-related cancellations for merchandise return features.

df_merchandise_returns = df_returns.loc[
    df_returns["ReturnCategory"] == "Merchandise"
].copy()

In [78]:
# Calculate the total value of merchandise returned or cancelled
# by each customer.

customer_merchandise_return_value = (
    df_merchandise_returns
    .groupby("CustomerID")["ReturnLineValue"]
    .sum()
    .reset_index(name="MerchandiseReturnValue")
)

In [79]:
customer_features = customer_features.merge(
    customer_merchandise_return_value,
    on="CustomerID",
    how="left"
)

customer_features["MerchandiseReturnValue"] = (
    customer_features["MerchandiseReturnValue"]
    .fillna(0)
)

In [80]:
customer_features["MerchandiseReturnValue"].describe().round(2)

count      4334.00
mean        107.96
std        2855.42
min           0.00
25%           0.00
50%           0.00
75%          14.78
max      168469.60
Name: MerchandiseReturnValue, dtype: float64

### Merchandise Return Value Rate

Merchandise Return Value Rate compares the value of merchandise returned or
cancelled with the value of merchandise purchased by each customer.

It is calculated as:

**Merchandise Return Value / Merchandise Monetary Value**

A value of 0 indicates no observed merchandise returns, while higher values
indicate that returned merchandise represents a larger proportion of the
customer's purchase value.

The rate is retained as a continuous ratio rather than being capped, because
unusually high values may represent genuine customer behaviour or transactions
whose corresponding original purchase occurred outside the available data
period.

In [81]:
# Calculate merchandise return value relative to merchandise purchase value.

customer_features["MerchandiseReturnValueRate"] = (
    customer_features["MerchandiseReturnValue"]
    / customer_features["MonetaryValue"]
)

In [82]:
# Confirm that the return-rate denominator is valid for every customer.

assert (customer_features["MonetaryValue"] > 0).all()

print("Monetary value denominator validation passed.")

Monetary value denominator validation passed.


In [83]:
customer_features[
    "MerchandiseReturnValueRate"
].describe().round(4)

count    4334.0000
mean        0.0211
std         0.0888
min         0.0000
25%         0.0000
50%         0.0000
75%         0.0098
max         2.3696
Name: MerchandiseReturnValueRate, dtype: float64

In [84]:
# Flag customers with at least one observed merchandise return.

customer_features["HasMerchandiseReturn"] = (
    customer_features["MerchandiseReturnValue"] > 0
).astype(int)

In [85]:
customer_features["HasMerchandiseReturn"].value_counts()

HasMerchandiseReturn
0    2828
1    1506
Name: count, dtype: int64

In [86]:
# Identify customers whose recorded merchandise return value exceeds
# their recorded merchandise purchase value.

high_return_rate_customers = customer_features.loc[
    customer_features["MerchandiseReturnValueRate"] > 1,
    [
        "CustomerID",
        "MonetaryValue",
        "MerchandiseReturnValue",
        "MerchandiseReturnValueRate"
    ]
].sort_values(
    "MerchandiseReturnValueRate",
    ascending=False
)

print(
    "Customers with return-value rate above 100%:",
    len(high_return_rate_customers)
)

high_return_rate_customers.head(10)

Customers with return-value rate above 100%: 2


,CustomerID,MonetaryValue,MerchandiseReturnValue,MerchandiseReturnValueRate
3795,17548.0,103.30,244.78,2.369603
3076,16546.0,787.15,883.08,1.121870


### Investigation of Return Rates Above 100%

Two customers have recorded merchandise return values greater than their
recorded merchandise purchase values.

These cases are reviewed before any treatment is applied, as they may reflect
returns relating to purchases made before the available transaction period
rather than data quality errors.

In [79]:
# Identify customers whose merchandise return value exceeds their
# recorded merchandise purchase value.

high_return_customer_ids = (
    high_return_rate_customers["CustomerID"]
    .tolist()
)

high_return_customer_ids

[17548.0, 16546.0]

In [80]:
# Review merchandise purchases for the customers with unusually high
# return-value rates.

high_return_purchases = (
    df_merchandise.loc[
        df_merchandise["CustomerID"].isin(high_return_customer_ids),
        [
            "CustomerID",
            "InvoiceNo",
            "InvoiceDate",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "LineValue"
        ]
    ]
    .sort_values(["CustomerID", "InvoiceDate"])
)

high_return_purchases

,CustomerID,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,LineValue
2488,16546.0,536663,2010-12-02 12:07:00,22867,HAND WARMER BIRD DESIGN,24,2.10,50.40
2489,16546.0,536663,2010-12-02 12:07:00,22633,HAND WARMER UNION JACK,24,2.10,50.40
2490,16546.0,536663,2010-12-02 12:07:00,22632,HAND WARMER RED RETROSPOT,24,2.10,50.40
2491,16546.0,536663,2010-12-02 12:07:00,22910,PAPER CHAIN KIT VINTAGE CHRISTMAS,40,2.55,102.00
2492,16546.0,536663,2010-12-02 12:07:00,22737,RIBBON REEL CHRISTMAS PRESENT,20,1.65,33.00
2493,16546.0,536663,2010-12-02 12:07:00,22952,60 CAKE CASES VINTAGE CHRISTMAS,24,0.55,13.20
60737,16546.0,544637,2011-02-22 11:20:00,22245,"HOOK, 1 HANGER ,MAGIC GARDEN",12,0.85,10.20
60738,16546.0,544637,2011-02-22 11:20:00,22251,BIRDHOUSE DECORATION MAGIC GARDEN,24,1.25,30.00
60739,16546.0,544637,2011-02-22 11:20:00,22250,DECORATION BUTTERFLY MAGIC GARDEN,16,0.85,13.60
60740,16546.0,544637,2011-02-22 11:20:00,22248,DECORATION PINK CHICK MAGIC GARDEN,16,0.85,13.60


In [81]:
# Review merchandise returns for the same customers.

high_return_returns = (
    df_merchandise_returns.loc[
        df_merchandise_returns["CustomerID"].isin(high_return_customer_ids),
        [
            "CustomerID",
            "InvoiceNo",
            "InvoiceDate",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "ReturnLineValue"
        ]
    ]
    .sort_values(["CustomerID", "InvoiceDate"])
)

high_return_returns

,CustomerID,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,ReturnLineValue
39,16546.0,C536812,2010-12-02 16:58:00,22578,WOODEN STAR CHRISTMAS SCANDINAVIAN,-36,0.85,30.60
40,16546.0,C536812,2010-12-02 16:58:00,22574,HEART WOODEN CHRISTMAS DECORATION,-192,0.72,138.24
41,16546.0,C536812,2010-12-02 16:58:00,22593,CHRISTMAS GINGHAM STAR,-144,0.72,103.68
42,16546.0,C536812,2010-12-02 16:58:00,22595,CHRISTMAS GINGHAM HEART,-144,0.72,103.68
43,16546.0,C536812,2010-12-02 16:58:00,22588,CARD HOLDER GINGHAM HEART,-192,2.10,403.20
44,16546.0,C536812,2010-12-02 16:58:00,22130,PARTY CONE CHRISTMAS DECORATION,-144,0.72,103.68
2,17548.0,C536391,2010-12-01 10:24:00,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65,19.80
3,17548.0,C536391,2010-12-01 10:24:00,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29,6.96
4,17548.0,C536391,2010-12-01 10:24:00,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29,6.96
5,17548.0,C536391,2010-12-01 10:24:00,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29,6.96


### Treatment of Return Rates Above 100%

Two customers have merchandise return values greater than their recorded
merchandise purchase values.

Transaction-level investigation indicates that these cases are explainable by
the available observation window rather than an obvious calculation error.

For one customer, a recorded purchase was subsequently returned in full, while
additional cancellation activity occurred at the beginning of the dataset and
has no corresponding purchase within the available history.

For the second customer, returned products do not appear among the customer's
recorded purchases, indicating that the original purchase may have occurred
before the available transaction period.

The Merchandise Return Value Rate is therefore retained without capping.
Values above 1 should be interpreted as returns exceeding purchases observed
within the dataset period, rather than as a literal lifetime return percentage.

Potential outlier treatment will be considered during feature preprocessing
rather than altering the underlying behavioural measure at this stage.


In [82]:
# Rename the feature to emphasise that the calculation is based only on
# purchase and return activity observed within the available dataset period.

customer_features = customer_features.rename(
    columns={
        "MerchandiseReturnValueRate":
        "ObservedMerchandiseReturnValueRate"
    }
)

## 9. Customer Feature Dataset Validation

Before feature preprocessing, the engineered customer-level dataset is reviewed
for structural integrity, missing values and customer uniqueness.

This validation ensures that feature engineering has not introduced duplicate
customer records or unexpected data-quality issues.

In [83]:
# Review the size and structure of the engineered customer-level dataset.

print("Customer feature rows:", f"{customer_features.shape[0]:,}")
print("Customer feature columns:", customer_features.shape[1])

customer_features.columns.tolist()

Customer feature rows: 4,334
Customer feature columns: 23


['CustomerID',
 'LastPurchaseDate',
 'Recency',
 'Frequency',
 'MonetaryValue',
 'AverageOrderValue',
 'UniqueProducts',
 'FirstPurchaseDate',
 'CustomerTenureDays',
 'TotalQuantity',
 'AverageQuantityPerOrder',
 'AverageDaysBetweenPurchases',
 'RepeatCustomer',
 'PostageSpend',
 'PostageInvoices',
 'PaidInvoices',
 'PostageInvoiceShare',
 'HasPostage',
 'ReturnInvoices',
 'ReturnValue',
 'MerchandiseReturnValue',
 'ObservedMerchandiseReturnValueRate',
 'HasMerchandiseReturn']

In [84]:
# Confirm that each customer appears exactly once in the feature dataset.

print(
    "Unique customers:",
    f"{customer_features['CustomerID'].nunique():,}"
)

duplicate_customers = (
    customer_features["CustomerID"]
    .duplicated()
    .sum()
)

print(
    "Duplicate CustomerID records:",
    duplicate_customers
)

Unique customers: 4,334
Duplicate CustomerID records: 0


In [85]:
assert len(customer_features) == customer_features["CustomerID"].nunique()

print("Customer uniqueness validation passed.")

Customer uniqueness validation passed.


In [86]:
# Identify missing values introduced during feature engineering.

missing_feature_summary = (
    customer_features
    .isna()
    .sum()
    .to_frame(name="MissingCount")
)

missing_feature_summary["MissingPercent"] = (
    missing_feature_summary["MissingCount"]
    / len(customer_features)
    * 100
).round(2)

missing_feature_summary = (
    missing_feature_summary[
        missing_feature_summary["MissingCount"] > 0
    ]
    .sort_values(
        "MissingPercent",
        ascending=False
    )
)

missing_feature_summary

,MissingCount,MissingPercent
AverageDaysBetweenPurchases,1505,34.73


### Missing-value validation

The only missing values in the engineered feature dataset occur in
`AverageDaysBetweenPurchases`.

These 1,505 missing values represent customers with only one observed purchase,
for whom an interval between successive purchases cannot be calculated.

The missingness is therefore structural and meaningful rather than a data
quality issue. No imputation is applied at the feature-engineering stage.

In [87]:
# Review the data types of the engineered customer features before
# preparing the dataset for modelling.

customer_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4334 entries, 0 to 4333
Data columns (total 23 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   CustomerID                          4334 non-null   float64       
 1   LastPurchaseDate                    4334 non-null   datetime64[ns]
 2   Recency                             4334 non-null   int64         
 3   Frequency                           4334 non-null   int64         
 4   MonetaryValue                       4334 non-null   float64       
 5   AverageOrderValue                   4334 non-null   float64       
 6   UniqueProducts                      4334 non-null   int64         
 7   FirstPurchaseDate                   4334 non-null   datetime64[ns]
 8   CustomerTenureDays                  4334 non-null   int64         
 9   TotalQuantity                       4334 non-null   int64         
 10  AverageQuantityPerOrder 

### Data type refinement

The engineered dataset contains valid feature values, but two fields require
minor semantic datatype refinement before modelling.

- `CustomerID` is an identifier and should be represented as an integer rather
  than a continuous numeric value.
- `PostageInvoices` is a count of unique invoices and should therefore be stored
  as an integer.

In [88]:
# Refine data types so that identifiers and count-based fields
# are represented consistently with their business meaning.

customer_features["CustomerID"] = (
    customer_features["CustomerID"]
    .astype("Int64")
)

customer_features["PostageInvoices"] = (
    customer_features["PostageInvoices"]
    .astype("int64")
)

In [89]:
customer_features[
    ["CustomerID", "PostageInvoices"]
].dtypes

CustomerID         Int64
PostageInvoices    int64
dtype: object

## 10. Model Feature Candidate Review

The engineered customer dataset contains both reference fields and behavioural
features.

Reference fields such as customer identifiers and transaction dates are retained
for traceability but should not be supplied directly to the clustering model.

The remaining engineered variables are reviewed as candidate clustering features
before preprocessing, transformation and scaling.

In [90]:
# Define fields retained for customer identification and traceability.
# These columns describe who the customer is or provide date context,
# but they are not direct clustering inputs.

reference_columns = [
    "CustomerID",
    "FirstPurchaseDate",
    "LastPurchaseDate"
]

reference_columns

['CustomerID', 'FirstPurchaseDate', 'LastPurchaseDate']

In [91]:
# Identify the behavioural variables that may be considered as clustering inputs.
# Final feature selection will follow redundancy and distribution analysis.

candidate_feature_columns = [
    "Recency",
    "Frequency",
    "MonetaryValue",
    "AverageOrderValue",
    "UniqueProducts",
    "CustomerTenureDays",
    "TotalQuantity",
    "AverageQuantityPerOrder",
    "AverageDaysBetweenPurchases",
    "RepeatCustomer",
    "PostageSpend",
    "PostageInvoices",
    "PaidInvoices",
    "PostageInvoiceShare",
    "HasPostage",
    "ReturnInvoices",
    "ReturnValue",
    "MerchandiseReturnValue",
    "ObservedMerchandiseReturnValueRate",
    "HasMerchandiseReturn"
]

In [92]:
# Create a temporary feature-only dataset for evaluating candidate
# clustering variables without modifying the full customer dataset.

feature_candidates = customer_features[
    candidate_feature_columns
].copy()

print("Candidate features:", feature_candidates.shape[1])

feature_candidates.head()

Candidate features: 20


,Recency,Frequency,MonetaryValue,AverageOrderValue,UniqueProducts,CustomerTenureDays,TotalQuantity,AverageQuantityPerOrder,AverageDaysBetweenPurchases,RepeatCustomer,PostageSpend,PostageInvoices,PaidInvoices,PostageInvoiceShare,HasPostage,ReturnInvoices,ReturnValue,MerchandiseReturnValue,ObservedMerchandiseReturnValueRate,HasMerchandiseReturn
0,326,1,77183.60,77183.600000,1,0,74215,74215.000000,NaN,0,0.0,0,1,0.0,0,1,77183.6,77183.6,1.0,1
1,2,7,4310.00,615.714286,103,365,2458,351.142857,60.839699,1,0.0,0,7,0.0,0,0,0.0,0.0,0.0,0
2,75,4,1437.24,359.310000,21,282,2332,583.000000,94.250926,1,360.0,4,4,1.0,1,0,0.0,0.0,0.0,0
3,19,1,1457.55,1457.550000,72,0,630,630.000000,NaN,0,300.0,1,1,1.0,1,0,0.0,0.0,0.0,0
4,310,1,294.40,294.400000,16,0,196,196.000000,NaN,0,40.0,1,1,1.0,1,0,0.0,0.0,0.0,0


## 11. Feature Redundancy Review

Candidate clustering features are reviewed for strong relationships before
finalising the engineered dataset.

Highly correlated features may represent similar aspects of customer behaviour.
Including several near-duplicate measures can cause those behaviours to receive
disproportionate influence during clustering.

Correlation is therefore used as a diagnostic rather than as an automatic
feature-removal rule. Business meaning will also be considered before deciding
which features to retain.

In [95]:
# Calculate pairwise correlations between the candidate behavioural features.
# Pandas automatically excludes missing observations on a pairwise basis.

feature_correlation = feature_candidates.corr(
    method="pearson"
)

feature_correlation.round(2)

,Recency,Frequency,MonetaryValue,AverageOrderValue,UniqueProducts,CustomerTenureDays,TotalQuantity,AverageQuantityPerOrder,AverageDaysBetweenPurchases,RepeatCustomer,PostageSpend,PostageInvoices,PaidInvoices,PostageInvoiceShare,HasPostage,ReturnInvoices,ReturnValue,MerchandiseReturnValue,ObservedMerchandiseReturnValueRate,HasMerchandiseReturn
Recency,1.00,-0.26,-0.12,0.00,-0.30,-0.51,-0.12,0.01,-0.02,-0.46,-0.05,-0.09,-0.26,-0.01,-0.03,-0.19,-0.00,-0.00,0.04,-0.20
Frequency,-0.26,1.00,0.55,0.02,0.69,0.48,0.56,0.01,-0.28,0.31,0.12,0.14,1.00,-0.01,0.06,0.72,0.07,0.04,0.01,0.31
MonetaryValue,-0.12,0.55,1.00,0.39,0.38,0.22,0.92,0.31,-0.13,0.13,0.18,0.13,0.55,0.00,0.06,0.38,0.40,0.37,0.08,0.17
AverageOrderValue,0.00,0.02,0.39,1.00,0.03,0.01,0.41,0.93,0.02,-0.00,0.03,0.01,0.02,0.01,0.02,0.03,0.92,0.92,0.24,0.06
UniqueProducts,-0.30,0.69,0.38,0.03,1.00,0.46,0.41,0.03,-0.21,0.34,0.19,0.13,0.69,-0.00,0.04,0.54,0.03,0.01,-0.03,0.28
CustomerTenureDays,-0.51,0.48,0.22,0.01,0.46,1.00,0.23,-0.00,0.25,0.72,0.06,0.14,0.48,-0.00,0.03,0.36,0.02,0.02,-0.00,0.39
TotalQuantity,-0.12,0.56,0.92,0.41,0.41,0.23,1.00,0.39,-0.13,0.13,0.16,0.15,0.56,0.00,0.06,0.38,0.38,0.36,0.09,0.17
AverageQuantityPerOrder,0.01,0.01,0.31,0.93,0.03,-0.00,0.39,1.00,0.02,-0.01,0.02,0.01,0.01,0.01,0.01,0.02,0.75,0.78,0.23,0.05
AverageDaysBetweenPurchases,-0.02,-0.28,-0.13,0.02,-0.21,0.25,-0.13,0.02,1.00,NaN,-0.05,-0.08,-0.28,-0.01,-0.04,-0.21,0.01,0.02,-0.02,-0.18
RepeatCustomer,-0.46,0.31,0.13,-0.00,0.34,0.72,0.13,-0.01,NaN,1.00,0.05,0.10,0.31,0.00,0.03,0.23,0.02,0.01,0.01,0.33


In [96]:
# Convert the correlation matrix into unique feature pairs and retain
# relationships with an absolute correlation of at least 0.80.

strong_correlations = (
    feature_correlation
    .where(
        np.triu(
            np.ones(feature_correlation.shape),
            k=1
        ).astype(bool)
    )
    .stack()
    .reset_index()
)

strong_correlations.columns = [
    "Feature1",
    "Feature2",
    "Correlation"
]

strong_correlations["AbsoluteCorrelation"] = (
    strong_correlations["Correlation"].abs()
)

strong_correlations = (
    strong_correlations.loc[
        strong_correlations["AbsoluteCorrelation"] >= 0.80
    ]
    .sort_values(
        "AbsoluteCorrelation",
        ascending=False
    )
    .reset_index(drop=True)
)

strong_correlations

,Feature1,Feature2,Correlation,AbsoluteCorrelation
0,Frequency,PaidInvoices,0.999657,0.999657
1,ReturnValue,MerchandiseReturnValue,0.971708,0.971708
2,PostageInvoiceShare,HasPostage,0.939682,0.939682
3,AverageOrderValue,AverageQuantityPerOrder,0.931465,0.931465
4,MonetaryValue,TotalQuantity,0.923607,0.923607
5,AverageOrderValue,MerchandiseReturnValue,0.920452,0.920452
6,AverageOrderValue,ReturnValue,0.915131,0.915131


### Redundancy review outcome

The correlation review identified several features containing substantially
overlapping information.

For clustering purposes:

- `PaidInvoices` is excluded because it is almost perfectly correlated with
  `Frequency`.
- `ReturnValue` is excluded in favour of `MerchandiseReturnValue`, which
  specifically represents product-related return activity.
- `HasPostage` is excluded in favour of the richer `PostageInvoiceShare`
  measure.
- `RepeatCustomer` is retained for interpretation but excluded from modelling
  because it is derived directly from `Frequency`.
- `HasMerchandiseReturn` is retained for interpretation but excluded from
  modelling because it is derived directly from `MerchandiseReturnValue`.

Other highly correlated measures, including monetary and quantity-based
features, are retained at this stage because they represent different business
behaviours. Their relationships will be reassessed after distribution and
outlier treatment during preprocessing.


In [97]:
# Define the behavioural features that will move forward to preprocessing.
# Redundant identifiers, binary derivatives and broader adjustment measures
# are retained in customer_features but excluded from the modelling matrix.

provisional_model_features = [
    "Recency",
    "Frequency",
    "MonetaryValue",
    "AverageOrderValue",
    "UniqueProducts",
    "CustomerTenureDays",
    "TotalQuantity",
    "AverageQuantityPerOrder",
    "AverageDaysBetweenPurchases",
    "PostageSpend",
    "PostageInvoices",
    "PostageInvoiceShare",
    "ReturnInvoices",
    "MerchandiseReturnValue",
    "ObservedMerchandiseReturnValueRate"
]

print("Provisional model features:", len(provisional_model_features))

Provisional model features: 15


## 12. Persist Engineered Customer Dataset

The validated customer-level feature dataset is persisted as Parquet so that
subsequent preprocessing and modelling stages do not depend on notebook state.

The full engineered dataset is retained rather than only the provisional model
features. This preserves identifiers, reference dates and interpretive fields
for traceability while allowing the preprocessing stage to select the final
clustering variables.

In [98]:
# Persist the complete engineered customer-level dataset in Parquet format.

customer_features_path = (
    "../data/processed/online_retail_customer_features.parquet"
)

customer_features.to_parquet(
    customer_features_path,
    engine="pyarrow",
    index=False
)

print(
    f"Customer feature dataset saved to: "
    f"{customer_features_path}"
)

Customer feature dataset saved to: ../data/processed/online_retail_customer_features.parquet


In [99]:
# Reload the persisted dataset to confirm that no rows or columns
# were lost during serialisation.

customer_features_check = pd.read_parquet(
    customer_features_path
)

print(
    "Original shape:",
    customer_features.shape
)

print(
    "Saved shape:",
    customer_features_check.shape
)

assert (
    customer_features_check.shape
    == customer_features.shape
)

print("Customer feature dataset validation passed.")

Original shape: (4334, 23)
Saved shape: (4334, 23)
Customer feature dataset validation passed.


In [100]:
customer_features_check[
    [
        "CustomerID",
        "FirstPurchaseDate",
        "LastPurchaseDate"
    ]
].dtypes

CustomerID                    Int64
FirstPurchaseDate    datetime64[ns]
LastPurchaseDate     datetime64[ns]
dtype: object

In [101]:
# Register the engineered customer feature dataset as a versioned
# Azure ML URI_FILE data asset.

customer_features_parquet_asset = Data(
    name="online-retail-customer-features-parquet",
    version="1",
    type=AssetTypes.URI_FILE,
    path=customer_features_path,
    description=(
        "Engineered customer-level features for the Online Retail "
        "customer segmentation project."
    )
)

registered_customer_features_parquet = ml_client.data.create_or_update(
    customer_features_parquet_asset
)

In [103]:
print("Name:", registered_customer_features_parquet.name)
print("Version:", registered_customer_features_parquet.version)
print("Type:", registered_customer_features_parquet.type)
print("Path:", registered_customer_features_parquet.path)

Name: online-retail-customer-features-parquet
Version: 1
Type: uri_file
Path: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/7f110ae40044adc4a2422decff6bf787d8bc44b91a78678bd4304cc45b4b42d9/online_retail_customer_features.parquet


In [104]:
# Create an MLTable definition that points to the registered
# customer-level feature dataset in Azure storage.

customer_features_table = mltable.from_parquet_files(
    paths=[
        {
            "file": registered_customer_features_parquet.path
        }
    ]
)

In [105]:
# Materialise the MLTable to confirm the engineered customer dataset
# can be read successfully.

customer_features_test = (
    customer_features_table
    .to_pandas_dataframe()
)

print("Customer features MLTable shape:", customer_features_test.shape)

Customer features MLTable shape: (4334, 23)


In [106]:
# Create a project folder for the engineered customer-features MLTable definition.
import os
customer_features_mltable_folder = "../data/mltable/customer-features"

os.makedirs(
    customer_features_mltable_folder,
    exist_ok=True
)

# Save the MLTable definition file.

customer_features_table.save(
    customer_features_mltable_folder
)

paths:
- file: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/7f110ae40044adc4a2422decff6bf787d8bc44b91a78678bd4304cc45b4b42d9/online_retail_customer_features.parquet
transformations:
- read_parquet:
    include_path_column: false
    path_column: Path
type: mltable

In [107]:
# Register the engineered customer feature MLTable as the governed
# input for downstream preprocessing and clustering.

customer_features_mltable_asset = Data(
    name="online-retail-customer-features",
    version="1",
    type=AssetTypes.MLTABLE,
    path=customer_features_mltable_folder,
    description=(
        "Customer-level engineered features for preprocessing "
        "and clustering in the Online Retail segmentation project."
    )
)

registered_customer_features_mltable = ml_client.data.create_or_update(
    customer_features_mltable_asset
)

In [109]:
print("Name:", registered_customer_features_mltable.name)
print("Version:", registered_customer_features_mltable.version)
print("Type:", registered_customer_features_mltable.type)

Name: online-retail-customer-features
Version: 1
Type: mltable
